In [13]:
import pystac_client

In [14]:
import sys
print(sys.executable)

c:\Users\Kurtis\Documents\Data Analysis\Projects\ecosystem-state-forecaster\.venv\Scripts\python.exe


In [15]:
# provisional small aoi over the sunshine coast region
# [min_lon, min_lat, max_lon, max_lat]

bbox = [152.80, -26.80, 152.90, -26.68]
time_range = "2023-01-01/2023-12-31"

In [16]:
# open the DEA stac catalogue
catalog = pystac_client.Client.open("https://explorer.dea.ga.gov.au/stac")

In [17]:
# search Sentinal-2A and 2B over our area of interest and time range

search = catalog.search(
    collections = ["ga_s2am_ard_3", "ga_s2bm_ard_3"],
    bbox = bbox,
    datetime = time_range
)

In [18]:
items = list(search.items())
print(f"Found {len(items)} Sentinel-2 scenes")

Found 71 Sentinel-2 scenes


In [19]:
from odc.stac import load, configure_rio

In [20]:
# configure  rasterio for access to DEA's public S3 bucket

configure_rio(cloud_defaults=True, aws={"aws_unsigned": True})

In [21]:
# build the cube from the scenes found in the search

ds = load(
    items,
    bands=["nbart_red", "nbart_nir_1"],
    bbox = bbox,
    crs = "EPSG:3577",  # GDA94 / Australian Albers - DEA's standard equal area CRS
    resolution = 10,    # metres per pixel
    groupby = "solar_day",  # merge scenes captured on the same day into one timestamp
    chunks={},          # {} means use Dask. Builds cube lazily, stream pixels on demand
)

In [22]:
ds

<xarray.Dataset> Size: 503MB
Dimensions:      (y: 1488, x: 1190, time: 71)
Coordinates:
  * y            (y) float64 12kB -3.057e+06 -3.057e+06 ... -3.072e+06
  * x            (x) float64 10kB 2.034e+06 2.034e+06 ... 2.046e+06 2.046e+06
  * time         (time) datetime64[us] 568B 2023-01-04T00:04:07.589441 ... 20...
    spatial_ref  int32 4B 3577
Data variables:
    nbart_red    (time, y, x) int16 251MB dask.array<chunksize=(1, 1488, 1190), meta=np.ndarray>
    nbart_nir_1  (time, y, x) int16 251MB dask.array<chunksize=(1, 1488, 1190), meta=np.ndarray>

In [23]:
# we cant average raw scenes, as most of those will have cloud somewhere and it disrupts the ndvi
# so, we mask first, using the quality layer

bands = ["nbart_red", "nbart_nir_1", "oa_fmask"],


In [24]:
# below we build the clear-pixel mask and compute the ndvi

# DEA fmask codes are: 0 = nodata, 1 = valid (i.e. clear), 2 = cloud, 3 = cloud shadow, 4 = snow, 5 = water
# we keep only clear land observations

clear = ds["oa_fmask"] == 1

#where (clear), keep clear pixels and set the rest to NaN

red = ds ["nbart_red"].where(clear)
nir = ds ["nbart_nir_1"].where(clear)

#NDVI = (NIR - Red) / (NIR + Red)
ndvi = (nir - red) / (nir + red)
ndvi.name = "ndvi"

KeyError: "No variable named 'oa_fmask'. Variables on the dataset include ['y', 'x', 'spatial_ref', 'time', 'nbart_red', 'nbart_nir_1']"